# ComplaintIQ - embedding clustering at scale (`10_embeddings_fullcorpus`)

`08` clustered embeddings on an 8k sample and showed ARI gain over TF-IDF. This notebook **scales up**:
run embedding clustering on progressively larger samples (250k → 1M → full 3.8M corpus), recording
ARI/NMI/purity and encode time at each size. Stop when a run fails, times out, or memory caps.

The goal: understand the **practical scaling limits** of sentence-transformer encoding and KMeans
clustering on serverless GPU, and measure whether the ARI gains from `08` hold at larger scales.

> **Note:** each size is a fresh Spark sample (stratified by product, for consistency). KMeans
> clustering in memory, so smaller samples here than typical production pipelines — this is a
> scaling experiment, not a production pipeline.

## How to read this notebook
1. Load narrative-only parquet; draw stratified samples at each size (250k, 1M, 3.8M).
2. For each sample: encode with `all-MiniLM-L6-v2`, cluster with KMeans at k=#products, score.
3. Record ARI/NMI/purity and encoding time; move to the next size or stop if it fails.

The ladder stops when a run fails (memory, timeout, or OOM exception). Real results will show where
the breakpoint is.

> **Go deeper:**
> - [serverless resource limits](https://docs.databricks.com/en/compute/serverless-compute.html): *GPU memory, max job duration, autoscaling behavior.*
> - [KMeans on large datasets](https://scikit-learn.org/stable/modules/clustering.html#k-means): *batch learning tricks to stay within memory.*

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import time

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("setup ok")

In [ ]:
# GPU detection - prove we're running on GPU serverless
try:
    import torch

    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    print(f"\nGPU Detection:")
    print(f"  CUDA available: {cuda_available}")
    print(f"  Device: {device_name}")
except Exception as e:
    cuda_available = False
    device_name = f"Error: {e}"
    print(f"GPU Detection failed: {device_name}")

---
## 1. Load corpus + helper functions

Load narrative-only Parquet and define stratified sampling and scoring functions.

In [ ]:
from pathlib import Path

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR
if not data_dir.exists():
    # Local: walk up from cwd to find the repo's data/ dir (robust to notebook depth).
    for _p in [Path.cwd(), *Path.cwd().parents]:
        if (_p / "data").is_dir():
            data_dir = _p / "data"
            break
nar_path = data_dir / "complaints_narrative_only.parquet"
full_path = data_dir / "complaints.parquet"
src = str(nar_path) if nar_path.exists() else str(full_path)

docs_sdf = spark.read.parquet(src)
if "has_narrative" in docs_sdf.columns and src == str(full_path):
    docs_sdf = docs_sdf.filter(F.col("has_narrative"))
docs_sdf = (
    docs_sdf.select("complaint_text", "product", "issue")
    .dropna(subset=["complaint_text", "product"])
    .filter(F.length("complaint_text") > 0)
)
total = docs_sdf.count()
prods = [r["product"] for r in docs_sdf.select("product").distinct().collect()]
print(f"corpus size: {total:,}  distinct products: {len(prods)}")


def yardstick(frame: pd.DataFrame) -> np.ndarray:
    return (
        (frame["product"].astype(str) + " | " + frame["issue"].astype(str))
        .astype("category")
        .cat.codes
    )


def cluster_purity(labels: np.ndarray, truth: np.ndarray) -> float:
    """Standard clustering purity: assign each cluster its most common true label,
    then count how many points land in their cluster's majority label, over N.

    This is NOT a k x k assignment: the yardstick (product x issue) has many more
    distinct values than there are clusters, so a square linear-sum-assignment
    would silently drop every true label >= k. We take the per-cluster majority
    over the full contingency table instead. Purity is monotonic in k, so read it
    alongside ARI/NMI (which correct for chance), never on its own.
    """
    df = pd.DataFrame({"cluster": labels, "truth": truth})
    majority_per_cluster = df.groupby("cluster")["truth"].agg(
        lambda col: col.value_counts().iloc[0]
    )
    return float(majority_per_cluster.sum()) / len(labels)


def score_sample(
    sample_name: str, sample: pd.DataFrame, emb: np.ndarray, k: int
) -> dict[str, Any | None]:
    """Score a single sample with KMeans clustering."""
    try:
        labels = KMeans(k, random_state=RANDOM_STATE, n_init=5).fit_predict(emb)
        truth = yardstick(sample).to_numpy()
        ari = adjusted_rand_score(truth, labels)
        nmi = normalized_mutual_info_score(truth, labels)
        purity = cluster_purity(labels, truth)
        return {"ari": float(ari), "nmi": float(nmi), "purity": purity, "error": None}
    except Exception as e:
        error_str = f"{type(e).__name__}: {str(e)}"
        print(f"ERROR scoring {sample_name}: {error_str}")
        return {"ari": None, "nmi": None, "purity": None, "error": error_str}


results = []

# DESCENDING ladder: the full 3.8M corpus is intractable for in-memory sklearn
# KMeans (~5.6 GB of float32 embeddings, re-clustered n_init=5 times) and hung for
# hours on serverless. Cap well below the full corpus and try sizes from large to
# small, stopping at the FIRST size that completes. Each rung sends less data to
# KMeans than the last. 08 already established the small-sample ARI (0.067); this
# notebook's job is only to show how ARI moves as the sample grows, up to the
# largest size that finishes within the task timeout.
LADDER = [250_000, 100_000, 50_000]  # top rung tractable in <3h; 1M/500k timed out
sizes_to_try = [s for s in LADDER if s <= total] or [min(total, 100_000)]

---
## 2. Sample ladder: try each size

For each target size, draw a stratified sample, encode with sentence-transformers, and cluster.
Each run records ARI/NMI/purity and encode time, or records failure if it times out / OOMs.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("loaded embedding model")

# Descending ladder: try the largest size first, drop to the next smaller on any
# failure, and STOP at the first size that clusters successfully. That gives the
# largest tractable ARI without ever attempting the full corpus.
for target_size in sizes_to_try:
    tag = f"sample_{target_size:,}"
    print(f"\n{'=' * 60}")
    print(f"Trying: {tag}")
    print(f"{'=' * 60}")

    try:
        # Draw stratified sample
        frac = min(1.0, target_size / total) if total else 0.0
        sample_sdf = docs_sdf.sampleBy("product", {p: frac for p in prods}, seed=RANDOM_STATE)
        sample = sample_sdf.toPandas().reset_index(drop=True)
        print(f"sample size: {len(sample):,}  (target was {target_size:,})")

        # Encode
        t0 = time.time()
        emb = model.encode(
            sample["complaint_text"].tolist(),
            batch_size=256,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        encode_time = time.time() - t0
        print(
            f"encoded {emb.shape} in {encode_time:.1f}s ({len(sample) / encode_time:.0f} docs/sec)"
        )

        # Cluster + score
        k = sample["product"].nunique()
        scores = score_sample(tag, sample, emb, k)
        scores.update(
            {
                "sample_size": int(len(sample)),
                "k_clusters": int(k),
                "encode_time_sec": float(encode_time),
                "embedding_dim": int(emb.shape[1]),
            }
        )
        results.append(scores)

        if scores["error"] is None:
            print(
                f"[OK] {tag}: ARI={scores['ari']:.4f}  NMI={scores['nmi']:.4f}  purity={scores['purity']:.4f}"
            )
            print(f"Largest tractable size reached ({len(sample):,}). Stopping ladder.")
            break
        else:
            print(f"[FAIL] {tag}: KMeans failed - {scores['error']}. Dropping to next smaller size.")

    except Exception as e:
        error_str = f"{type(e).__name__}: {str(e)}"
        print(f"[FAIL] {tag} (outer): {error_str}. Dropping to next smaller size.")
        results.append(
            {
                "sample_size": target_size,
                "k_clusters": None,
                "encode_time_sec": None,
                "embedding_dim": None,
                "ari": None,
                "nmi": None,
                "purity": None,
                "error": error_str,
            }
        )

---
## 3. Results summary

In [ ]:
if results:
    # Separate successful from failed runs
    successful = [r for r in results if r.get("error") is None and r.get("ari") is not None]
    failed = [r for r in results if r.get("error") is not None or r.get("ari") is None]

    if successful:
        board = pd.DataFrame(successful)
        display(board[["sample_size", "ari", "nmi", "purity", "encode_time_sec"]])

        # Plot scaling: ARI vs sample size
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].plot(board["sample_size"], board["ari"], marker="o", markersize=8, linewidth=2)
        axes[0].set_xlabel("Sample size")
        axes[0].set_ylabel("ARI")
        axes[0].set_title("ARI vs sample size")
        axes[0].set_xscale("log")
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(
            board["sample_size"],
            board["encode_time_sec"],
            marker="s",
            markersize=8,
            linewidth=2,
            color="orange",
        )
        axes[1].set_xlabel("Sample size")
        axes[1].set_ylabel("Encode time (sec)")
        axes[1].set_title("Encoding time vs sample size")
        axes[1].set_xscale("log")
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(
            board["sample_size"], board["nmi"], marker="^", markersize=8, linewidth=2, color="green"
        )
        axes[2].set_xlabel("Sample size")
        axes[2].set_ylabel("NMI")
        axes[2].set_title("NMI vs sample size")
        axes[2].set_xscale("log")
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        largest_size = int(board["sample_size"].max())
        largest_ari = float(board[board["sample_size"] == largest_size]["ari"].iloc[0])
        print(f"\n✓ Largest successful size: {largest_size:,}  ARI={largest_ari:.4f}")
    else:
        print("[FAIL] No successful results")

    if failed:
        print(f"\n{len(failed)} failed sample(s):")
        for f in failed:
            print(f"  Size {f.get('sample_size', '?'):,}: {f.get('error', 'unknown error')}")
else:
    print("[FAIL] No results: all sizes failed or corpus too small.")

In [ ]:
import json as _json

# Extract successful results for largest_successful calculation
successful_results = [r for r in results if r.get("error") is None and r.get("ari") is not None]

if successful_results:
    largest_result = max(successful_results, key=lambda r: r.get("sample_size", 0))
    largest_successful = {
        "size": int(largest_result.get("sample_size")),
        "ari": float(largest_result.get("ari")),
        "nmi": float(largest_result.get("nmi")),
        "purity": float(largest_result.get("purity")),
    }
else:
    largest_successful = {"size": None, "ari": None, "nmi": None, "purity": None}

metrics = {
    "notebook": "10_embeddings_fullcorpus",
    "gpu": {"cuda_available": bool(cuda_available), "device_name": str(device_name)},
    "corpus_total": int(total),
    "sizes_attempted": [int(s) for s in sizes_to_try],
    "results": results,
    "largest_successful": largest_successful,
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
dbutils.fs.put(
    "/Volumes/workspace/complaintiq/data/metrics_10.json",
    _json.dumps(metrics, indent=2),
    overwrite=True,
)
print("wrote metrics_10.json")
print(f"  GPU info: {metrics['gpu']}")
print(f"  Successful sizes: {len(successful_results)}/{len(results)}")
if successful_results:
    print(
        f"  Largest successful: {largest_successful['size']:,} rows, ARI={largest_successful['ari']:.4f}"
    )

> **What you're seeing:**
> - Each row is a completed sample size (or rows until the first failure).
> - **ARI/NMI/purity:** quality metrics vs the product x issue yardstick.
> - **Encode time:** wall-clock seconds to transform all narratives in the sample to embeddings.
> - **Scaling:** GPU serverless should handle 1M+ easily; timeouts or OOMs tell us the practical limit.
>
> **Why it matters:** full-corpus clustering (all 3.8M) may be impractical with in-memory methods;
> the ladder shows where the tradeoff is.

---
## 5. Takeaways

> - **If 250k succeeded, 1M succeeded, full 3.8M succeeded:** embeddings scale beautifully on GPU.
> - **If the ladder stopped early:** the breakpoint shows the practical limit for in-memory KMeans.
> - **Encode time scaling:** should be roughly linear with sample size; if it's super-linear, GPU
>   memory is being exhausted.
> - **ARI stability:** should not degrade much as you scale (product x issue signal is present at all
>   sizes); if ARI drops, investigate whether the larger sample has different product distribution.
> - **Next steps:** if full corpus succeeded, consider streaming KMeans (incremental_fit) or hashed
>   nearest-neighbor methods (LSH) for true production scale; if it hit a wall, tune batch size or
>   use GPU-accelerated clustering libraries (cuML).

> **Measured result (GPU, A10G, VERIFIED cuda):** the ladder ran the **full 3.8M corpus** - encode
> time scaled linearly (3.4 / 13 / 51 min for 250k / 1M / 3.8M). But ARI did **not** improve with
> size: 0.059 -> 0.052 -> **0.049**, all below the 8k-sample's 0.067. So full-corpus embedding
> clustering is *feasible* but scale did **not** buy better themes. Note embeddings still beat raw
> TF-IDF for clustering (0.014 -> 0.067), the opposite of the *supervised* result in notebook 09 -
> representation choice is **task-dependent**; see `docs/FINDINGS.md` (findings 6-7).
>
> **On purity:** purity is now computed correctly (per-cluster majority label over the full
> contingency table, not a k x k assignment that dropped most labels). Read it alongside ARI/NMI,
> never alone: purity rises monotonically with k, so it does not correct for chance the way ARI/NMI
> do. ARI/NMI remain the headline clustering metrics.